# Additional LC3 analysis columns


## Imports

In [ ]:
import os

import numpy as np
import pandas as pd
from skimage.measure import label, regionprops
from skimage.segmentation import relabel_sequential

## Parameters

Add folder names to `IMAGE_FOLDER_NAMES`; each one is processed independently and gets its own pair of CSVs.

In [ ]:
RESULTS_FOLDER = "Results"
MASK_FOLDER = os.path.join(RESULTS_FOLDER, "masks")

IMAGE_FOLDER_NAMES = [
    "L3 Lipophagy images",
]

SOURCE_SUFFIX = "LC3_lipophagy_mitophagy_analysis.csv"   # "<folder>_<SOURCE_SUFFIX>"
OUTPUT_SUFFIX = "Additional_Analysis"                    # "<folder>_<OUTPUT_SUFFIX>_per_cell.csv"

MFI_COLUMNS = ["Green_MFI_raw", "Yellow_MFI_raw", "Red_MFI_raw"]   # raw-intensity, not in the masks
NEW_COLUMNS = ["N_Green_Circles_Surrounding_LD",
               "N_Green_Circles_Surrounding_Mito",
               "N_Green_Circles_Surrounding_LD_or_Mito"]

## Analysis

In [ ]:
def coloc_green_with(green_objects, target_mask):
    """§5's helper: each green object overlapping or encasing the target counts once, so one
    mito streak crossing two LC3 rings counts as 2. Returns (n_structures, overlap_area_px)."""
    n = 0
    coloc = np.zeros(green_objects.shape, dtype=bool)
    for p in regionprops(green_objects):
        comp = green_objects == p.label
        if np.any(comp & target_mask):
            n += 1
            coloc[comp] = True
    return n, int((coloc & target_mask).sum())


def measure_cell(cell_mask, green_objects, green_struct, red, mito):
    """Every per-cell mask metric for one green-positive cell, read back from saved masks.

    The original columns use §6's definitions verbatim. A green object is a circle when
    fill_green_rings enclosed an interior for it -- that interior is `green_filled &
    ~green_struct`, i.e. the ring flag §6 counted -- and it surrounds a target when that
    interior holds target pixels."""
    # fill_green_rings already labeled and split these, so restrict and relabel rather than
    # re-labeling a bool mask: 8-connectivity would diagonally re-merge what it just split.
    objs = np.where(cell_mask, green_objects, 0)
    objs, _, _ = relabel_sequential(objs)
    red_in = red & cell_mask
    mito_in = mito & cell_mask

    n_struct = int(objs.max())                  # dots + rings
    n_ring = n_ld = n_mito = n_either = 0
    for p in regionprops(objs):
        lumen = (objs == p.label) & ~green_struct    # empty for a solid dot
        if not lumen.any():
            continue
        n_ring += 1
        around_ld = bool((red_in & lumen).any())
        around_mito = bool((mito_in & lumen).any())
        n_ld += around_ld
        n_mito += around_mito
        n_either += around_ld or around_mito

    green_area = int((objs > 0).sum())          # footprint (rings filled to disks)
    n_gr, area_gr = coloc_green_with(objs, red_in)
    n_gb, area_gb = coloc_green_with(objs, mito_in)

    return {
        # original analysis columns, recomputed from the masks
        "Cell_Area_px": int(cell_mask.sum()),
        "N_Green_Dots_Circles_Per_Cell": n_struct,
        "N_Green_Structures": n_struct,
        "N_Green_Dots": n_struct - n_ring,
        "N_Green_Rings": n_ring,
        "Green_Dots_Circles_Area_px": green_area,
        "Green_Structure_Area_px": green_area,
        "N_Lipid_Droplets": int(label(red_in).max()),
        "LipidDroplet_Area_px": int(red_in.sum()),
        "Mito_Area_px": int(mito_in.sum()),
        "N_GreenRed_Coloc_Lipophagy": n_gr,
        "GreenRed_Overlap_Area_px": area_gr,
        "N_GreenBlue_Coloc_Mitophagy": n_gb,
        "GreenBlue_Overlap_Area_px": area_gb,
        # additional columns
        "N_Green_Circles_Surrounding_LD": n_ld,
        "N_Green_Circles_Surrounding_Mito": n_mito,
        "N_Green_Circles_Surrounding_LD_or_Mito": n_either,
    }


def collect_from_masks(mask_root):
    """One row per green-positive cell for every image folder under mask_root."""
    rows = []
    for stem in sorted(os.listdir(mask_root)):
        image_dir = os.path.join(mask_root, stem)
        if not os.path.isdir(image_dir):
            continue

        def load(name):
            return np.load(os.path.join(image_dir, name))

        labeled_cells = load("labeled_cells.npy")
        green_objects = load("green_objects.npy")
        green_struct = load("green_struct.npy")
        red = load("red_lipid.npy")
        mito = load("mito.npy")
        for cid in load("green_positive_ids.npy"):
            row = {"Filename": f"{stem}.czi", "Cell_ID": int(cid)}
            row.update(measure_cell(labeled_cells == cid, green_objects, green_struct, red, mito))
            rows.append(row)
    return pd.DataFrame(rows)


def verify_against_original(measured, original, source_name):
    """Assert every shared column reproduces the shipped CSV exactly, else the new columns
    describe other cells."""
    check = original.merge(measured, on=["Filename", "Cell_ID"], suffixes=("_orig", "_new"))
    assert len(check) == len(original) == len(measured), \
        f"{source_name}: mask rows do not line up with the CSV"
    shared = [c for c in original.columns
              if c in measured.columns and c not in ("Filename", "Cell_ID")]
    for c in shared:
        assert (check[f"{c}_orig"] == check[f"{c}_new"]).all(), f"{source_name}: {c} differs"
    print(f"  mask readback matches {source_name} on {len(shared)} shared column(s)")


def summarize_by_condition(per_cell):
    """n, mean and sample SD per metric for CH vs noCH."""
    metrics = [c for c in per_cell.columns if c not in ("Filename", "Condition", "Cell_ID")]
    summary = []
    for metric in metrics:
        row = {"Metric": metric}
        for cond in ("CH", "noCH"):
            values = per_cell.loc[per_cell["Condition"] == cond, metric]
            row[f"{cond}_n_cells"] = int(values.count())
            row[f"{cond}_mean"] = round(float(values.mean()), 2)
            row[f"{cond}_SD"] = round(float(values.std(ddof=1)), 2)
        summary.append(row)
    return pd.DataFrame(summary)


def analyze_folder(folder_name):
    """Measure one image folder's masks, verify, and write its two CSVs."""
    mask_root = os.path.join(MASK_FOLDER, folder_name)
    source_csv = os.path.join(RESULTS_FOLDER, f"{folder_name}_{SOURCE_SUFFIX}")
    if not os.path.isdir(mask_root):
        print(f"[skip] no masks at '{mask_root}'\n")
        return None
    if not os.path.isfile(source_csv):
        print(f"[skip] no original CSV at '{source_csv}'\n")
        return None

    print(f"{folder_name}: START")
    measured = collect_from_masks(mask_root)
    original = pd.read_csv(source_csv)
    verify_against_original(measured, original, os.path.basename(source_csv))

    # MFI is measured on the raw uint16 channels, which the masks do not carry.
    per_cell = measured.merge(original[["Filename", "Cell_ID"] + MFI_COLUMNS],
                              on=["Filename", "Cell_ID"])

    per_cell.insert(1, "Condition",
                    np.where(per_cell["Filename"].str.contains("_noCH_"), "noCH", "CH"))
    per_cell["Condition"] = pd.Categorical(per_cell["Condition"], ["CH", "noCH"], ordered=True)
    per_cell = per_cell.sort_values(["Condition", "Filename", "Cell_ID"]).reset_index(drop=True)
    per_cell["Condition"] = per_cell["Condition"].astype(str)

    front = ["Filename", "Condition", "Cell_ID"]
    per_cell = per_cell[front + [c for c in original.columns if c not in front] + NEW_COLUMNS]

    stem = os.path.join(RESULTS_FOLDER, f"{folder_name}_{OUTPUT_SUFFIX}")
    per_cell.to_csv(f"{stem}_per_cell.csv", index=False)
    summarize_by_condition(per_cell).to_csv(f"{stem}_summary_by_condition.csv", index=False)
    print(f"  -> {len(per_cell)} green-positive cell row(s)")
    for suffix in ("per_cell", "summary_by_condition"):
        print(f"  -> wrote {stem}_{suffix}.csv")
    print()
    return per_cell


last = None
for folder_name in IMAGE_FOLDER_NAMES:
    result = analyze_folder(folder_name)
    if result is not None:
        last = result

last.head() if last is not None else None